In [ ]:
!git clone https://github.com/rhasspy/piper

In [ ]:
cd ./piper/src/python

In [ ]:
!pip install "pip<24.1"
!pip install -U datasets
!pip install huggingface_hub

In [ ]:
# For Nvidia GPU, use CUDA
!pip install -r requirements.txt -r requirements_dev.txt --extra-index-url https://download.pytorch.org/whl/cu124

In [ ]:
# For AMD GPU, use ROCm
!pip install -r requirements.txt -r requirements_dev.txt --extra-index-url https://download.pytorch.org/whl/rocm6.1

In [ ]:
from huggingface_hub import hf_hub_download
datapack = hf_hub_download(repo_id="thennal/IMaSC", repo_type="dataset", filename="data/train-00000-of-00011-e9c81350c905a174.parquet")

In [ ]:
import pyarrow.parquet as pq
ds = pq.read_table( datapack )
ds_v = ds

In [ ]:
# prompt: Using parquet table ds_v : iterate over each rows and , extract audio  into "ds_raw/wav" directory with filename format indexnumber.wav . Also, create a ds_raw/metadata.csv file with two columns index and text . Use "|" as separator and no header row

import os

os.makedirs("ds_raw/wav", exist_ok=True)

with open("ds_raw/metadata.csv", "w") as metadata_file:
    for index, row in enumerate(ds_v.to_pylist()):
        # Assuming the audio data is in a column named 'audio'
        # and the text is in a column named 'text'
        # You might need to adjust these column names based on your parquet file schema
        audio_data = row['audio']['bytes']
        text_data = row['text']

        wav_filename = f"ds_raw/wav/{index}.wav"
        with open(wav_filename, "wb") as wav_file:
            wav_file.write(audio_data)

        metadata_file.write(f"{index}|{text_data}\n")

In [ ]:
!head ds_raw/metadata.csv
!file ds_raw/wav/0.wav
!ls -lh ds_raw/wav/0.wav

In [ ]:
!cpuinfo

In [ ]:
cd ../src/python

In [ ]:
!python3 -m piper_train.preprocess \
--language ml \
--input-dir ds_raw/ \
--output-dir ds_traindata \
--dataset-format ljspeech \
--single-speaker \
--sample-rate 22050

In [ ]:
from huggingface_hub import hf_hub_download
checkpoing_file = hf_hub_download(repo_id="rhasspy/piper-checkpoints", repo_type="dataset", filename="ml/ml_IN/meera/medium/epoch=3168-step=92368.ckpt")

In [ ]:
cd ./piper_train/vits/monotonic_align

In [ ]:
!mkdir -p monotonic_align
!cythonize -i core.pyx
!ls -l
!mv *.so monotonic_align/

In [ ]:
cd ../../../

In [ ]:
%load_ext tensorboard

In [ ]:
!mkdir -p ./ds_traindata
%tensorboard --logdir ./ds_traindata

In [ ]:
!git pull

In [ ]:
# Training . Batch-size should be adjusted as per GPU RAM

!python3 -m piper_train \
--dataset-dir ds_traindata \
--accelerator 'gpu' \
--devices 1 \
--batch-size 16 \
--validation-split 0.0 \
--num-test-examples 0 \
--max_epochs 3800 \
--resume_from_checkpoint "{checkpoing_file}" \
--checkpoint-epochs 1 \
--precision 32 --save_last 1

In [ ]:
# inference. 

!python3 -m piper_train.infer \
--sample-rate 22050 \
--input_file ./ml-sample-long.jsonl \
--checkpoint ./ds_traindata/lightning_logs/version_0/checkpoints/last.ckpt \
--output-dir ./wav-output 